In [1]:
import os
import shutil
from pathlib import Path
from dotenv import load_dotenv
import psycopg2
from sqlalchemy import create_engine
from minio import Minio
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from delta import configure_spark_with_delta_pip

In [2]:
load_dotenv()

is_docker = os.path.exists('/.dockerenv')

POSTGRES_HOST = os.getenv("POSTGRES_HOST", "localhost") if is_docker else "localhost"
POSTGRES_PORT = os.getenv("POSTGRES_PORT", "5432")
POSTGRES_DB   = os.getenv("POSTGRES_DB",   "streaming")
POSTGRES_USER = os.getenv("POSTGRES_USER", "streaming")
POSTGRES_PASS = os.getenv("POSTGRES_PASSWORD", "streaming123")

MINIO_HOST = "minio:9000" if is_docker else "localhost:9000"
MINIO_USER = os.getenv("MINIO_ROOT_USER",     "minioadmin")
MINIO_PASS = os.getenv("MINIO_ROOT_PASSWORD", "minioadmin")

engine = create_engine(
    f"postgresql+psycopg2://{POSTGRES_USER}:{POSTGRES_PASS}@{POSTGRES_HOST}:{POSTGRES_PORT}/{POSTGRES_DB}"
)

builder = (
    SparkSession.builder
    .appName("gold-para-postgres")
    .config("spark.sql.extensions",
            "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog",
            "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.hadoop.fs.defaultFS", "file:///")
)
spark = configure_spark_with_delta_pip(builder).getOrCreate()

cliente_minio = Minio(
    MINIO_HOST, access_key=MINIO_USER, secret_key=MINIO_PASS, secure=False
)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/21 19:13:23 WARN Utils: Your hostname, DESKTOP-S3JB25M, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/06/21 19:13:23 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/mnt/c/Users/felip/www/university/eng-dados/projeto-ed-final/.venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/felipe/.ivy2.5.2/cache
The jars for the packages stored in: /home/felipe/.ivy2.5.2/jars
io.delta#delta-spark_4.1_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-f1bff75d-10bd-4bd6-947f-7380b09b6ac7;1.0
	confs: [default]
	found io.delta#delta-spark_4.1_2.13;4.1.0 in central
	found io.delta#delta-storage;4.1.0 in central
	found io.unitycatalog#unitycatalog-client;0.4.0 in central
	found org.slf

In [3]:
conn = psycopg2.connect(
    host=POSTGRES_HOST, port=POSTGRES_PORT, dbname=POSTGRES_DB,
    user=POSTGRES_USER, password=POSTGRES_PASS
)
conn.autocommit = True
with conn.cursor() as cur:
    cur.execute("CREATE SCHEMA IF NOT EXISTS gold;")
conn.close()
print("Schema 'gold' pronto no Postgres.")

Schema 'gold' pronto no Postgres.


In [4]:
def download_delta_do_minio(bucket: str, prefix: str,
                            cliente: Minio, local_dir: Path) -> None:
    if local_dir.exists():
        shutil.rmtree(local_dir)
    local_dir.mkdir(parents=True)
    objetos = list(cliente.list_objects(bucket, prefix=f"{prefix}/", recursive=True))
    print(f"  {len(objetos)} arquivo(s) em {bucket}/{prefix}/")
    for obj in objetos:
        relative = obj.object_name[len(prefix) + 1:]
        destino = local_dir / relative
        destino.parent.mkdir(parents=True, exist_ok=True)
        cliente.fget_object(bucket, obj.object_name, str(destino))


def escreve_no_postgres(df, tabela: str) -> None:
    pdf = df.toPandas()
    pdf.to_sql(tabela, engine, schema="gold", if_exists="replace", index=False)
    print(f"  gold.{tabela} → {len(pdf):,} registros escritos no Postgres.")

## Dimensões

In [5]:
DIMENSOES = [
    "dim_tempo",
    "dim_usuario",
    "dim_plano",
    "dim_artista",
    "dim_musica",
]

for dim in DIMENSOES:
    print(f"\n{'=' * 55}")
    print(f"Espelhando: {dim}")
    print(f"{'=' * 55}")

    local_dir = Path(f"/tmp/gold_local/{dim}")
    download_delta_do_minio("gold", f"dimensoes/{dim}", cliente_minio, local_dir)

    df = spark.read.format("delta").load(str(local_dir))
    escreve_no_postgres(df, dim)

print("\nDimensões espelhadas com sucesso.")


Espelhando: dim_tempo
  6 arquivo(s) em gold/dimensoes/dim_tempo/


26/06/21 19:13:43 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


  gold.dim_tempo → 1,096 registros escritos no Postgres.

Espelhando: dim_usuario
  6 arquivo(s) em gold/dimensoes/dim_usuario/


  gold.dim_usuario → 3,000 registros escritos no Postgres.

Espelhando: dim_plano
  6 arquivo(s) em gold/dimensoes/dim_plano/


  gold.dim_plano → 3 registros escritos no Postgres.

Espelhando: dim_artista
  6 arquivo(s) em gold/dimensoes/dim_artista/


  gold.dim_artista → 500 registros escritos no Postgres.

Espelhando: dim_musica
  6 arquivo(s) em gold/dimensoes/dim_musica/
  gold.dim_musica → 10,000 registros escritos no Postgres.

Dimensões espelhadas com sucesso.


## Fatos

In [ ]:
print(f"\n{'=' * 55}")
print("Espelhando: fato_reproducao")
print(f"{'=' * 55}")

local_dir = Path("/tmp/gold_local/fato_reproducao")
download_delta_do_minio("gold", "fatos/fato_reproducao", cliente_minio, local_dir)

df_repro = spark.read.format("delta").load(str(local_dir))
escreve_no_postgres(df_repro, "fato_reproducao")

In [7]:
print(f"\n{'=' * 55}")
print("Espelhando: fato_pagamento")
print(f"{'=' * 55}")

local_dir = Path("/tmp/gold_local/fato_pagamento")
download_delta_do_minio("gold", "fatos/fato_pagamento", cliente_minio, local_dir)

df_pag = spark.read.format("delta").load(str(local_dir))
escreve_no_postgres(df_pag, "fato_pagamento")

print("\nEspelho Gold → Postgres concluído.")


Espelhando: fato_pagamento
  78 arquivo(s) em gold/fatos/fato_pagamento/


  gold.fato_pagamento → 10,500 registros escritos no Postgres.

Espelho Gold → Postgres concluído.


In [8]:
spark.stop()